# ⚙️ Multi-Provider LLM Routing & Infrastructure Selection Matrix

**My Objective:** I am architecting a multi-provider LLM routing matrix to automatically select the optimal model endpoint based on deterministic constraints (data sovereignty, TTFT latency limits, and tokenomics). I built this evaluation to engineer resilient, cost-effective inference pipelines across three distinct production environments: Public FAQ, PII Document Analysis, and Live Agent Assist.

In [ ]:
import sys; sys.path[:0] = [".", "module01", "../module01"]
import os
from landscape_utils import *
from tokens_utils import chat_client, ollama_client

ollama = ollama_client()
try:
    groq = chat_client()                      # Groq API
except Exception as e:                        
    groq = None
    print("no hosted client:", str(e)[:80])
    
GROQ_MODEL = os.getenv("MODEL")
print("hosted model:", GROQ_MODEL, "| expected EMI:", EXPECTED_EMI, "| expected ratio:", EXPECTED_RATIO, "%")

## 1. Inference Infrastructure Catalog & VRAM Economics
I loaded an indicative catalog of available models to establish a baseline for context windows, token pricing (USD per million tokens), and operational topologies (hosted API, local compute, or self-hosted GPU).

In [ ]:
cat = load_catalog()
print(f"{'model':46} {'access':11} {'open':5} {'ctx':>8} {'in$':>6} {'out$':>6} {'Q':>2} {'s':>4}")
for m in cat:
    pin = "?" if m["price_in"] is None else f"{m['price_in']:.3f}"
    pout = "?" if m["price_out"] is None else f"{m['price_out']:.2f}"
    print(f"{m['name'][:46]:46} {m['access']:11} {str(m['open_weight'])[:1]:5} {m['ctx_tokens']:8d} {pin:>6} {pout:>6} {m['quality']:2d} {m['latency_s']:4.1f}")

OVERRIDES = {
    "groq-llama31-8b": {"price_out": 0.08},      
}

**Cost-Scaling Analysis — On-Premise GPU Economics:** When I run open-weight models locally or on-premise, token generation is technically free. However, the operational baseline is never zero; our ongoing compute expenses simply shift from variable API usage to fixed monthly capital expenditures (CapEx) for physical hardware or leased cloud GPUs.

## 2. Workload Simulation & Pipeline Telemetry
I modeled the monthly financial impact of running these models against three distinct production workloads:
* **FAQ:** High-volume (20,000 calls/day), low-complexity, no sensitive data.
* **Documents:** Moderate volume (2,000 calls/day), heavy token payloads, contains Customer PII (Data must remain air-gapped/in-house).
* **Agent Assist:** High volume (30,000 calls/day), strict sub-3-second Time to First Token (TTFT) latency limit.

In [ ]:
for key, sc in SCENARIOS.items():
    print(f"\n{key}: {sc['title']}  ({sc['calls_per_day']:,} calls/day, {sc['in_tokens']:,} in / {sc['out_tokens']} out per call)")
    for m in cat:
        m = {**m, **OVERRIDES.get(m["id"], {})}
        inr = monthly_cost_inr(m, sc)
        print(f"   {m['name'][:46]:46} " + ("price unknown" if inr is None else f"Rs {inr:>12,}/month"))

**LLMOps Perspective — Premium Tier API Waste:** I calculated that for basic FAQ workloads, the frontier flagship model costs 281 times more than `gpt-oss-20b`. This massive compute premium is financially unjustifiable for low-complexity tasks. I recommend strict semantic routing: we must only trigger flagship models for high-risk, high-reasoning workloads where pipeline failures result in significant financial loss.

## 3. API Telemetry & Latency Profiling (TTFT vs. TPS)
Theoretical catalog numbers are insufficient for production engineering. I sent the same prompt to Groq and Ollama to capture live telemetry, directly benchmarking Time to First Token (TTFT) against aggregate Tokens Per Second (TPS).

In [ ]:
print(PROMPT, "\n")
RESULTS = {}
for label, client, model in [("groq", groq, GROQ_MODEL), ("ollama", ollama, "llama3.2:3b")]:
    if client is None:
        print(f"--- {label}: skipped (no client - add your key to .env)\n"); continue
    try:
        r = time_stream(client, model)
        r["check"] = check_answer(r["text"])
        RESULTS[label] = r
        print(f"--- {label}: {model}")
        print(f"first token {r['ttft_s']} s | total {r['total_s']} s | {r['completion_tokens']} tokens | {r['tokens_per_s']} tokens/s")
        print("arithmetic check:", r["check"])
        print(r["text"].strip()[:500], "\n")
    except Exception as e:
        print(f"--- {label}: skipped ({type(e).__name__}: {str(e)[:120]})\n")

MEASURED = {
    "groq-gpt-oss-20b":   {"latency_s": 1.4, "quality": 4},
    "ollama-llama32-3b":  {"latency_s": 9.0, "quality": 2},
}
OVERRIDES.update(MEASURED)

**Performance Benchmark — Defining UX Latency:** I observed that `gpt-oss-20b` on Groq achieved a significantly faster Time to First Token (TTFT), even if other tiers generated more tokens per second overall. For asynchronous ETL pipelines, TPS is the priority. However, for live conversational UI, I concluded that TTFT is the absolute limiting factor for retaining customer attention and preventing session abandonment.

## 4. Deterministic Routing Logic & Constraint Filters
I aggressively filtered models using hard architectural constraints (e.g., PII leakage, TTFT limits). I then ranked the surviving endpoints using a weighted selection algorithm balancing cost, throughput speed, and syntactic quality.

In [ ]:
for key, sc in SCENARIOS.items():
    ranked, excluded = rank(sc, overrides=OVERRIDES)
    print(f"\n=== {key}: {sc['title']}   weights {sc['weights']}")
    if not ranked:
        print("   nobody survives the hard constraints - relax one or add a model")
    for i, m in enumerate(ranked, 1):
        print(f"  {i}. {m['name'][:46]:46} score {m['score']:5.1f}   Rs {m['cost_inr']:>10,}/month   {m['latency_s']} s   Q{m['quality']}")
    for name, why in excluded:
        print(f"  x  {name[:46]:46} {'; '.join(why)}")

**Architectural Insight — Air-Gapped Deployments & Data Sovereignty:** Because the 'documents' workload contains PII, I recognized we cannot route payloads to third-party flagship APIs. To prevent PII leakage, I was forced to deploy an open-weight model on-premises. Guaranteeing data sovereignty requires deliberately sacrificing the faster TTFT and advanced reasoning capabilities of cloud endpoints in favor of isolated inference.

## 5. Final Deployment Decisions
Based on my hard constraints and matrix scoring, I have designated the routing architecture as follows:

* **Public FAQ Chatbot:** `gpt-oss-20b` on Groq. Because FAQ tasks have low complexity, I found the open-weight model provides sufficient quality. I avoided the flagship model entirely to eliminate unnecessary costs.
* **Document Analysis (PII):** `gpt-oss-20b` self-hosted on a smaller GPU server. I deployed this strictly on local infrastructure to ensure data privacy and isolated inference.
* **Live Agent Assist:** `gpt-oss-20b` on Groq. I chose this specifically because it is optimized for low TTFT and cost-efficient inference, fitting safely within our strict latency budgets.

In [ ]:
# Output matrix data to markdown file for CI/CD tracking
from pathlib import Path
import landscape_utils
OUT = Path(landscape_utils.__file__).with_name("s10_matrix_output.md")   
lines = ["# Model Selection Matrix Output\n", f"expected EMI {EXPECTED_EMI}, ratio {EXPECTED_RATIO}%\n"]
for key, sc in SCENARIOS.items():
    ranked, excluded = rank(sc, overrides=OVERRIDES)
    lines.append(f"\n## {key}: {sc['title']}")
    for i, m in enumerate(ranked, 1):
        lines.append(f"{i}. {m['name']} - score {m['score']} - Rs {m['cost_inr']:,}/month - {m['latency_s']} s - Q{m['quality']}")
OUT.write_text("\n".join(lines), encoding="utf-8")
print("Matrix analysis saved to", OUT)